<a href="https://colab.research.google.com/github/FilletMinion/OCR-RAG-pipeline/blob/main/Task_9_1_4_FINAL_RAG_%2B_OCR_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PPT link: https://docs.google.com/presentation/d/13Vs9jRvBqQ-DhPkHbFw-UYiXBlurCwc_CbD5s_c6ZsE/edit?usp=sharing

In [ ]:
!pip install llama-index llama-index-llms-huggingface transformers accelerate bitsandbytes
!pip install -q torch
!pip install -q llama-index-embeddings-huggingface
!pip install PyMuPDF
!pip install easyocr pillow

# Check CUDA version first
!nvcc --version

# Install llama-cpp-python with CUDA 12.x support
!pip install --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

In [ ]:
!pip install llama-index-llms-llama-cpp

In [ ]:
from google.colab import files
import fitz  # PyMuPDF
import os
import torch
from typing import List
import time
import json

import easyocr
from PIL import Image
import io
import numpy as np

from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import VectorStoreIndex, Document, get_response_synthesizer, SimpleDirectoryReader,ServiceContext, Settings
from llama_index.core.retrievers import VectorIndexRetriever, QueryFusionRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

**Uploading functions**


In [ ]:
def load_pdf_with_pymupdf(pdf_path: str) -> List[Document]:
    reader = easyocr.Reader(['en'], gpu=False)  # Set gpu=False if no GPU available
    """Load a PDF and convert it to LlamaIndex Document format using PyMuPDF."""
    # Open the PDF
    doc = fitz.open(pdf_path)

    # Extract text from each page
    documents = []

    for i, page in enumerate(doc):

        text = page.get_text("text", sort=True)

        lines = text.splitlines()
        non_empty_lines = [line for line in lines if line.strip()]
        text = os.linesep.join(non_empty_lines)

        # If we can't detect text with PyMuPDF, try OCR
        if not text.strip():
            # Convert page to image (higher DPI = better quality but slower)
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # 2x zoom = 144 DPI
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            img_array = np.array(img)
            results = reader.readtext(img_array)

            for detection in results:
                img_text = detection[1]  # detection[0] is bbox, detection[1] is text, detection[2] is confidence
                confidence = detection[2]
                text = text + " " + img_text

            if not text.strip():
                continue

        # Create Document object with metadata
        documents.append(
            Document(
                text=text,
                metadata={
                    "file_name": os.path.basename(pdf_path),
                    "page_number": i + 1,
                    "total_pages": len(doc)
                }
            )
        )

    # Close the document
    doc.close()

    # Print stats
    print(f"Processed {pdf_path}:")
    print(f"Extracted {len(documents)} pages with content")

    return documents

In [ ]:
# Download Mistral model if not already present
model_path = "/content/mistral.gguf"
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

In [ ]:
# Configure the LLM
llm = LlamaCPP(
    model_path="/content/mistral.gguf",
    temperature=0.3,
    max_new_tokens=256,
    context_window=2048,
    model_kwargs={"n_threads": 2}
)

# Set as the default LLM and embedding model
Settings.llm = llm

# Initialize embedding model
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.embed_model = embed_model

# create semantic chunker
semantic_splitter = SemanticSplitterNodeParser(embed_model=embed_model)

In [ ]:
# Function to create a query engine that uses query expansion
def create_query_engine(index, top_k = 2):
    """Create a query engine that uses query expansion."""
    # First create multiple retrievers (base retriever)
    base_retriever = index.as_retriever(similarity_top_k= top_k)

    # Create the query engine with the fusion retriever
    query_engine = RetrieverQueryEngine.from_args(
        retriever = base_retriever,
        llm=llm,
        verbose=True
      )

    return query_engine


In [ ]:
doc_types = ["Resume", "Contract", "Lenders Fee Sheet", "ID", "PaySlip", "Disclosure", "Mortgage Agreement", "Letter", "Other"]


def chunk_and_index(pdf_path: str) -> List[Document]:
    """Load a PDF and convert it to LlamaIndex Document format using PyMuPDF."""
    # Open the PDF
    start_time = time.time()
    doc = fitz.open(pdf_path)

    # Extract text from each page
    documents = []
    summary = {
        "File" : os.path.basename(pdf_path),
        "Pages" : len(doc),
        "Documents Found" : 0,
        "Chunks" : 0,
        "Doc Types" : set(),
        "Time (seconds)" : 0,

        "Page Previews" : []
        }


    for i, page in enumerate(doc):

        text = page.get_text("text", sort=True)
        lines = text.splitlines()
        non_empty_lines = [line for line in lines if line.strip()]
        text = os.linesep.join(non_empty_lines)

        # Skip empty pages
        if not text.strip():
            continue
        summary["Documents Found"] += 1
        cur_doc = [Document(text=text)]
        index = VectorStoreIndex.from_documents(cur_doc)
        query_engine = create_query_engine(index)

        prompt = f"""
        What kind of document do you think this is?
        Choose from this list options and ONLY reply with your choice: {doc_types}

        Example 1:
        Resume

        Example 2:
        Other

        Example 3:
        Contract
        """

        doc_type = str(query_engine.query(prompt))

        # hard extracts the doc type from the repsonse incase the LLM rambles
        for name in doc_types:
          if name.lower() in doc_type.lower():
            doc_type = name
            break

        print("doc type:", doc_type, i)

        page_chunks = semantic_splitter.get_nodes_from_documents(cur_doc)
        print("found", len(page_chunks), "chunks")
        summary["Chunks"] += len(page_chunks)
        set.add(summary["Doc Types"], doc_type)


        for chunk_count, chunk in enumerate(page_chunks):

            documents.append(
                Document(
                    text = chunk.text,
                    metadata={
                        "file_name": os.path.basename(pdf_path),
                        "page_number": i + 1,
                        "doc_type" : doc_type,
                        "chunk_number" : chunk_count + 1

                    }
                )
            )
        summary["Page Previews"].append(f"""Page {i}: {doc_type} """)

    summary["Doc Types"] = list(summary["Doc Types"])
    # Close the document
    doc.close()
    end_time = time.time()
    summary["Time (seconds)"] = round(end_time - start_time, 2)
    # Print stats
    print(f"Processed {pdf_path}:")
    print(f"Extracted {len(documents)} pages with content")

    return documents, summary

In [ ]:
def predict_doc_type(query, selection = doc_types):
  prompt = f"""
  You are an AI assistant in a RAG (Retrieval Augmented Generation) system tasked with identifying the type of document needed to answer a user's query.

  The user's query is: {query}

  Based on this query, what kind of document would contain the information necessary to answer it?
  Choose from this list options and ONLY reply with your choice: {selection}

  Example 1:
  Resume

  Example 2:
  Other

  Example 3:
  Contract
  """

  input_doc = [Document(text=prompt)]
  index = VectorStoreIndex.from_documents(input_doc)
  query_engine = create_query_engine(index)
  doc_type = str(query_engine.query(prompt))

  # hard extracts the doc type from the repsonse incase the LLM rambles
  for name in selection:
    if name.lower() in doc_type.lower():
      return name

  return "Other"


## Gradio Chatbot

In [ ]:
import gradio as gr

chatbot_data = {
    "filepath" : None,
    "documents" : None,
    "types" : [],
    "target_type" : "Other",
    "metadata" : None,
    "autoquery" : True,
    "totalchunks" : 1,
    "top_k" : 1
}

def respond(message, history):
  start_time = time.time()
  doc_type = chatbot_data["target_type"]
  if chatbot_data["autoquery"] == True:
    doc_type = predict_doc_type(message, chatbot_data["types"])

  if doc_type == None:
    return "I am not confident enough to answer this prompt with my current settings."

  filtered_docs = []
  sources = []
  for doc in chatbot_data["documents"]:
    if doc.metadata["doc_type"] == doc_type:
      filtered_docs.append(doc)

  index = VectorStoreIndex.from_documents(filtered_docs)
  query_engine = create_query_engine(index, chatbot_data["top_k"])
  response = query_engine.query(message)
  source_nodes = response.source_nodes
  end_time = time.time()

  answer = str(response) + "\n\nSources: "
  for node in response.source_nodes:
      page = node.node.metadata["page_number"]
      chunk = node.node.metadata["chunk_number"]
      doc_type = node.node.metadata["doc_type"]
      confidence = round(node.score, 3) * 100
      answer = answer + "\nPage " + str(page) + " (" + doc_type +"), Chunk " + str(chunk) + ", Confidence: " + str(confidence) + "%"
  answer = answer + "\n\nTime taken: " + str(round(end_time - start_time, 2)) + " seconds"
  return answer

def process_pdf(file_path):
  documents, summary = chunk_and_index(file_path)
  chatbot_data["types"] = []
  chatbot_data["types"] = summary["Doc Types"]
  chatbot_data["metadata"] = summary
  chatbot_data["totalchunks"] = summary["Chunks"]
  chatbot_data["documents"] = documents
  chatbot_data["target_type"] = summary["Doc Types"][0]

  return (
      chatbot_data["metadata"],
      gr.Dropdown(choices = chatbot_data["types"],
                  value=chatbot_data["types"][0] if chatbot_data["types"] else None,
                  interactive = not chatbot_data["autoquery"]),
      gr.Slider(0, chatbot_data["totalchunks"], label = "Chunks", value = 1, interactive = True, step = int),
      gr.Column(visible=True, scale = 1)

  )

def load_pdf(file_path):
  chatbot_data["filepath"] = file_path
  return gr.Button("Process PDF", variant="primary", interactive = True)

def toggle_autoquery(setting):
  chatbot_data["autoquery"] = setting
  return gr.Dropdown(
                label ="Drop Down",
                choices = chatbot_data["types"],
                value = chatbot_data["types"][0] if chatbot_data["types"] else None,
                interactive = not chatbot_data["autoquery"]
            )

def change_top_k(new_k):
  chatbot_data["top_k"] = new_k

def manual_doc_type(doc_type):
  chatbot_data["target_type"] = doc_type

# Create the Gradio interface
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# 📄 RAG-Powered Chatbot")

    with gr.Row():
        with gr.Column(scale=0.5):
            pdf_input = gr.File(
                label="Upload PDF to get started",
                file_types=[".pdf"],
                type="filepath"
            )
            process_pdf_button = gr.Button("Process PDF", variant="primary", interactive = False)

            gr.Markdown("# Settings")

            drop_down = gr.Dropdown(
                label="Drop Down",
                choices=[],
                interactive= False
            )

            auto_query_box = gr.Checkbox(label="Auto Query", value = True)

            chunk_slider = gr.Slider(0, 10, label = "Chunks", value = 5, interactive = False, step = int)
            # Create a JSON outputwindow
            result_display = gr.JSON(label="PDF Data")

        with gr.Column(scale = 1, visible= False) as chat_container:
            chatbot = gr.ChatInterface(
                fn=respond,
            )

    # Connect the button to process the PDF
    process_pdf_button.click(
        fn=process_pdf,
        inputs=pdf_input,
        outputs=[result_display, drop_down, chunk_slider, chat_container]
    )

    pdf_input.change(
        fn=load_pdf,
        inputs=pdf_input,
        outputs = process_pdf_button
    )

    auto_query_box.change(
        fn=toggle_autoquery,
        inputs=auto_query_box,
        outputs=drop_down
    )

    chunk_slider.change(
        fn = change_top_k,
        inputs = chunk_slider,
        outputs = None
    )

    drop_down.change(
        fn=manual_doc_type,
        inputs=drop_down,
        outputs=None
    )

    app.launch()

/tmp/ipython-input-2484795960.py:84: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:
/usr/local/lib/python3.12/dist-packages/gradio/layouts/column.py:59: UserWarning: 'scale' value should be an integer. Using 0.5 will cause issues.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6d93e6fdebf891a1eb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
